# Find genes linked to a trait from a high-throughput CRISPR screen
## A three-month research project • Google Colab / Jupyter
**Question:** Which gene perturbations change abundance in a trait-selected population relative to a control?

This notebook assumes a **pooled CRISPR knockout/CRISPRi screen with sgRNA count data**. It does not analyse microscopy plates, RNA-seq expression, or single-cell Perturb-seq. If your screen uses another assay, its analysis needs a different design.

**Start here:** Open this `.ipynb` in https://colab.research.google.com/ using **File → Upload notebook**, then select **Runtime → Run all**. A CPU runtime is sufficient. The default run uses simulated data and needs no uploads. Set `USE_DEMO=False` when you have real data.

**What you receive:** sample QC, PCA, guide effect sizes, exploratory gene statistics, a candidate shortlist, optional MAGeCK analysis/import, optional pathway enrichment, and a downloadable results ZIP.

**Evidence level:** simulated hits are examples only. Real screen hits are candidate associations requiring independent validation; enrichment alone does not prove that a gene causes the trait. This project does not require a supervised AI model. PCA supports pattern exploration without inventing training labels.

## Three-month plan (12 weeks)
| Weeks | Work | Deliverable |
|---|---|---|
| 1–2 | Specify trait, species, perturbation type, comparison, biological replicates; select public data or obtain count tables | Research question and dataset provenance |
| 3–4 | Check library annotation, count quality, replicate identity, and confounding | QC report and frozen analysis plan |
| 5–6 | Filter, normalize, inspect guide effects and exploratory gene rankings | Reproducible exploratory analysis |
| 7–8 | Run MAGeCK; assess control guides, guide concordance and alternative thresholds | Direction-specific gene results |
| 9–10 | Interpret pathways and review literature with matched gene identifiers | Candidate evidence table |
| 11–12 | Prepare validation proposals, figures, report and repository | Final report, notebook and results |

Record the intended contrast before seeing hit lists. For drug resistance, enrichment may indicate resistance-promoting loss of function; depletion may reflect sensitivity or a general growth defect. For sorted high/low trait bins, interpretation depends on which bin is called treatment. Include a matched untreated endpoint and, where available, time-zero data to separate trait effects from general fitness.

## 1. Packages and project settings
Installation runs only if dependencies are missing. Exact versions used are saved in the results. For long-term reproduction, use that environment record. Colab sessions are temporary: download outputs before closing.

In [ ]:
import importlib.util, subprocess, sys
requirements = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
                "matplotlib": "matplotlib", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from pathlib import Path
from IPython.display import display
import json, hashlib, platform, shutil, warnings
sns.set_theme(style="whitegrid")
USE_DEMO = True
TRAIT = "Example selected trait — replace with your actual phenotype"
SPECIES = "Specify organism and gene identifier system"
PAIRED = False  # True requires matched biological pairs in metadata column 'pair'
MIN_CONTROL_COUNT = 30
MIN_CONTROL_FRACTION = 2/3
MIN_GUIDES_PER_GENE = 3
FDR_THRESHOLD = 0.05
MIN_ABS_LOG2FC = 0.75
MIN_GUIDE_CONCORDANCE = 0.75
SEED = 42
OUT = Path("screen_results_demo" if USE_DEMO else "screen_results_real")
OUT.mkdir(exist_ok=True)
print("SIMULATED EXAMPLE" if USE_DEMO else "REAL DATA", "|", TRAIT)

## 2. Input files
Upload **raw integer counts**, not normalized counts, CPM, or log counts.

`counts.csv` (or `.tsv`) has columns `sgRNA`, `Gene`, then one column per sample. sgRNA IDs must be unique and each row must map to a single gene.

| sgRNA | Gene | control_1 | control_2 | control_3 | treatment_1 | treatment_2 | treatment_3 |
|---|---|---:|---:|---:|---:|---:|---:|
| guide_001 | GENE_A | 520 | 480 | 510 | 120 | 140 | 100 |

`metadata.csv` contains `sample`, `group` (`control` or `treatment`), `replicate` (independent biological unit), and optionally `batch`, `pair`.
Technical replicates must first be summed within each biological sample. At least three biological replicates per condition are required for the exploratory tests here. Paired samples share `pair`, not independent identities within a condition. Multiple batches require a model with batch terms; this simple baseline stops rather than ignoring them.

Optional `control_guides.csv`: one column `sgRNA`, containing validated neutral guide IDs, used for control-based normalization. Non-targeting controls may differ from targeting guides in growth screens; choose controls justified for your assay. Without this file, median-ratio normalization assumes most guides are unaffected.

In [ ]:
def read_table(path):
    return pd.read_csv(path, sep="\t" if str(path).endswith((".tsv", ".txt")) else ",")
control_ids = []
truth = None
if USE_DEMO:
    rng = np.random.default_rng(SEED)
    genes = [f"GENE_{i:03d}" for i in range(1, 201)]
    gene_labels = np.repeat(genes, 4)
    guide_ids = [f"sg_{i:04d}" for i in range(len(gene_labels))]
    neutral_ids = [f"NTC_{i:03d}" for i in range(60)]
    labels = np.r_[gene_labels, ["NON_TARGETING"]*len(neutral_ids)]
    control_ids = neutral_ids
    samples = [f"control_{i}" for i in range(1,4)] + [f"treatment_{i}" for i in range(1,4)]
    metadata = pd.DataFrame({"sample":samples, "group":["control"]*3+["treatment"]*3,
                             "replicate": ["C1","C2","C3","T1","T2","T3"], "batch":"one"})
    truth = pd.DataFrame({"Gene":genes, "true_log2fc":0.0})
    truth.loc[:9, "true_log2fc"] = 2.0
    truth.loc[10:19, "true_log2fc"] = -2.0
    true_map = truth.set_index("Gene").true_log2fc.to_dict()
    effect = np.array([true_map.get(g,0) for g in labels]) * rng.uniform(.75,1.1,len(labels))
    abundance = rng.lognormal(np.log(550), .45, len(labels))
    counts = pd.DataFrame({"sgRNA":guide_ids+neutral_ids, "Gene":labels})
    for j,s in enumerate(samples):
        gene_noise = {g:rng.normal(0,.10) for g in sorted(set(labels))}
        noise = np.array([gene_noise[g] for g in labels])
        mean = abundance * 2**(noise + (effect if j>=3 else 0)) * rng.uniform(.8,1.2)
        dispersion = .012
        counts[s] = rng.negative_binomial(1/dispersion, (1/dispersion)/(1/dispersion+mean))
    truth.to_csv(OUT/"simulation_truth.csv",index=False)
else:
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded)
    count_names = [f for f in names if Path(f).stem == "counts"]
    meta_names = [f for f in names if Path(f).stem == "metadata"]
    if len(count_names)!=1 or len(meta_names)!=1:
        raise ValueError("Upload one counts.csv/tsv and one metadata.csv/tsv.")
    counts, metadata = read_table(count_names[0]), read_table(meta_names[0])
    if "control_guides.csv" in names:
        control_ids = pd.read_csv("control_guides.csv")["sgRNA"].tolist()
display(counts.head(), metadata)

## 3. Validate the experimental design
Fix failures before proceeding. A replicate is an independent biological experiment, not a sequencing lane or another guide targeting the same gene.

In [ ]:
def validate_input(counts, metadata, control_ids, paired=False):
    if not {"sgRNA","Gene"}.issubset(counts): raise ValueError("Missing sgRNA/Gene columns.")
    if not {"sample","group","replicate"}.issubset(metadata): raise ValueError("Missing metadata columns.")
    if counts.empty or metadata.empty: raise ValueError("Empty input.")
    if counts[["sgRNA","Gene"]].isna().any().any(): raise ValueError("Missing guide/gene ID.")
    if counts.sgRNA.astype(str).str.strip().eq("").any() or counts.Gene.astype(str).str.strip().eq("").any():
        raise ValueError("Blank identifiers.")
    if counts.sgRNA.duplicated().any(): raise ValueError("Duplicate sgRNA IDs.")
    if metadata.isna().any().any(): raise ValueError("Missing metadata values.")
    if metadata['sample'].duplicated().any(): raise ValueError("Duplicate sample names.")
    if set(metadata.group)!={"control","treatment"}: raise ValueError("Groups must be control/treatment.")
    if metadata.groupby('group').size().min()<3: raise ValueError("Need >=3 biological samples per group.")
    if metadata.duplicated(["group","replicate"]).any(): raise ValueError("Aggregate technical replicates first.")
    sample_cols = [c for c in counts if c not in ["sgRNA","Gene"]]
    if set(sample_cols)!=set(metadata['sample']): raise ValueError("Metadata/count sample columns differ.")
    if "batch" in metadata and metadata.batch.nunique()>1:
        raise ValueError("Multiple batches: use an appropriate covariate-aware model, e.g. MAGeCK MLE.")
    x = counts[sample_cols].apply(pd.to_numeric, errors='raise').to_numpy(dtype=float)
    if not np.isfinite(x).all() or (x<0).any() or not np.equal(x,np.floor(x)).all():
        raise ValueError("Counts must be finite nonnegative integers.")
    if (x.sum(axis=0)==0).any(): raise ValueError("A sample has no reads.")
    if len(set(control_ids)) != len(control_ids): raise ValueError("Duplicate control IDs.")
    if not set(control_ids).issubset(set(counts.sgRNA)): raise ValueError("Unknown control guide IDs.")
    if paired:
        if "pair" not in metadata: raise ValueError("Paired design requires pair column.")
        tab = pd.crosstab(metadata['pair'],metadata.group)
        if not (tab==1).all().all(): raise ValueError("Each pair needs exactly one sample per group.")
    return list(metadata['sample'])
samples = validate_input(counts,metadata,control_ids,PAIRED)
counts = counts.copy()
counts['sgRNA'] = counts.sgRNA.astype(str)
counts['Gene'] = counts.Gene.astype(str)
control_ids = list(map(str,control_ids))
raw = counts.set_index('sgRNA')[samples].astype(float)
annotation = counts.set_index('sgRNA')['Gene']
ctrl = metadata.loc[metadata.group=='control','sample'].tolist()
trt = metadata.loc[metadata.group=='treatment','sample'].tolist()
print(f"Validated {len(raw)} guides and {len(samples)} biological samples.")

## 4. Count quality and replicate consistency
Check sample depth, missing guides, Gini inequality, and correlation. These are diagnostics, not universal pass/fail thresholds. Check library coverage, known essential genes and mapped-read statistics from the upstream counting pipeline separately. A count table alone cannot reveal the fraction of FASTQ reads that mapped.

In [ ]:
def gini(x):
    a = np.sort(np.asarray(x,dtype=float))
    return (2*np.sum(np.arange(1,len(a)+1)*a)/(len(a)*a.sum())-(len(a)+1)/len(a)) if a.sum() else np.nan
qc = pd.DataFrame({"reads":raw.sum(), "zero_fraction":(raw==0).mean(),
                   "median_count":raw.median(), "gini":raw.apply(gini)})
display(qc)
qc.to_csv(OUT/'sample_qc.csv')
fig,axes=plt.subplots(1,2,figsize=(12,4))
qc.reads.plot.bar(ax=axes[0], title='Read depth'); axes[0].set_ylabel('Counts')
sns.boxplot(data=np.log2(raw+1),ax=axes[1]); axes[1].tick_params(axis='x',rotation=60)
axes[1].set_title('Raw guide-count distributions'); axes[1].set_ylabel('log2(count + 1)')
plt.tight_layout(); plt.savefig(OUT/'qc_counts.png',dpi=160); plt.show()
plt.figure(figsize=(7,6))
sns.heatmap(np.log2(raw+1).corr(method='spearman'),annot=True,vmin=0,vmax=1,cmap='viridis')
plt.title('Guide abundance correlations (can be driven by baseline abundance)')
plt.tight_layout(); plt.savefig(OUT/'sample_correlations.png',dpi=160); plt.show()

## 5. Filter guides and normalize
Retain guides meeting the baseline-count threshold in the required fraction of controls. Do not filter using treatment effects, which could remove real depleted guides.

Size factors use guide-wise geometric means and sample-wise median ratios. If neutral guides are supplied, use them as the normalization reference; otherwise use all retained guides. Reference guides must have positive counts in every sample. Widespread selection may violate median normalization assumptions. These exploratory factors are not claimed to reproduce MAGeCK exactly.

In [ ]:
keep = (raw[ctrl]>=MIN_CONTROL_COUNT).mean(axis=1)>=MIN_CONTROL_FRACTION
filtered = raw.loc[keep].copy()
if len(filtered)<10: raise ValueError("Too few guides after filtering.")
ref_ids = filtered.index.intersection(control_ids) if control_ids else filtered.index
reference = filtered.loc[ref_ids]
reference = reference.loc[(reference>0).all(axis=1)]
if len(reference)<10: raise ValueError("Fewer than 10 valid normalization guides; review controls/counts.")
geomean = np.exp(np.log(reference).mean(axis=1))
size_factors = reference.div(geomean,axis=0).median(axis=0)
size_factors /= np.exp(np.log(size_factors).mean())
normalized = filtered.div(size_factors,axis=1)
logcounts = np.log2(normalized+1)
print('Retained',len(filtered),'of',len(raw),'guides; normalization references:',len(reference))
display(size_factors.rename('size_factor').to_frame())
size_factors.to_csv(OUT/'normalization_size_factors.csv')
normalized.to_csv(OUT/'normalized_guide_counts.csv')
pca_model = PCA(n_components=2)
coords = pca_model.fit_transform(logcounts.T)
pc = metadata.set_index('sample').loc[samples].reset_index()
pc['PC1'],pc['PC2'] = coords[:,0],coords[:,1]
plt.figure(figsize=(7,5)); sns.scatterplot(data=pc,x='PC1',y='PC2',hue='group',s=100)
for _,r in pc.iterrows(): plt.text(r.PC1,r.PC2,r['sample'],fontsize=9)
plt.xlabel(f'PC1 ({pca_model.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca_model.explained_variance_ratio_[1]:.1%})')
plt.title('PCA of normalized guide counts'); plt.tight_layout()
plt.savefig(OUT/'pca.png',dpi=160); plt.show()

## 6. Exploratory gene analysis
For each gene and biological sample, take the median log2 normalized guide count. Compare those **sample-level** values using Welch's t-test, or a paired t-test for matched pairs. Guides are not treated as independent biological replicates. Adjust p-values across eligible genes with Benjamini–Hochberg.

This is a transparent exploratory baseline, **not a count-aware CRISPR inference model**. Its log-scale assumptions and p-values may be unreliable with small samples; final claims should use MAGeCK or a suitable model for your design. Minimum effect size and guide agreement help prioritization but are not an additional statistical proof. Do not describe the selected candidates' realized error rate as guaranteed by the nominal FDR threshold.

In [ ]:
def bh_adjust(p):
    p = np.asarray(p,dtype=float)
    if not np.isfinite(p).all() or ((p<0)|(p>1)).any(): raise ValueError("Invalid p-values.")
    order=np.argsort(p); ranks=np.arange(1,len(p)+1)
    adjusted=np.minimum.accumulate((p[order]*len(p)/ranks)[::-1])[::-1]
    out=np.empty(len(p)); out[order]=np.clip(adjusted,0,1)
    return out

guide = pd.DataFrame({'Gene':annotation.loc[filtered.index],
                      'log2fc':logcounts[trt].mean(axis=1)-logcounts[ctrl].mean(axis=1)})
guide['is_control'] = guide.index.isin(control_ids)
target = guide.loc[~guide.is_control]
gene_counts = logcounts.loc[target.index].groupby(target.Gene).median()
num_guides = target.groupby('Gene').size()
gene_counts = gene_counts.loc[num_guides[num_guides>=MIN_GUIDES_PER_GENE].index]
if gene_counts.empty: raise ValueError('No genes have enough retained guides.')
if PAIRED:
    pairs = metadata['pair'].drop_duplicates()
    ccols = [metadata.loc[(metadata['pair']==p)&(metadata.group=='control'),'sample'].iloc[0] for p in pairs]
    tcols = [metadata.loc[(metadata['pair']==p)&(metadata.group=='treatment'),'sample'].iloc[0] for p in pairs]
    result = stats.ttest_rel(gene_counts[tcols],gene_counts[ccols],axis=1)
else:
    result = stats.ttest_ind(gene_counts[trt],gene_counts[ctrl],axis=1,equal_var=False)
pvals = np.asarray(result.pvalue)
if not np.isfinite(pvals).all():
    warnings.warn('Undefined tests (e.g. zero variance) assigned p=1; inspect affected genes.')
pvals = np.where(np.isfinite(pvals),pvals,1.0)
genes = pd.DataFrame({'Gene':gene_counts.index,'n_guides':num_guides.loc[gene_counts.index].values,
                     'log2fc':(gene_counts[trt].mean(axis=1)-gene_counts[ctrl].mean(axis=1)).values,
                     'p_value_exploratory':pvals, 'q_value_exploratory':bh_adjust(pvals)})
median_guide = target.groupby('Gene').log2fc.median()
agreement = target.groupby('Gene').log2fc.apply(lambda x: np.mean(np.sign(x)==np.sign(x.median())))
genes['median_guide_log2fc'] = genes.Gene.map(median_guide)
genes['guide_concordance'] = genes.Gene.map(agreement)
genes['direction'] = np.where(genes.log2fc>0,'enriched','depleted')
genes['candidate_exploratory'] = ((genes.q_value_exploratory<=FDR_THRESHOLD)&
    (genes.log2fc.abs()>=MIN_ABS_LOG2FC)&(genes.guide_concordance>=MIN_GUIDE_CONCORDANCE)&
    (np.sign(genes.log2fc)==np.sign(genes.median_guide_log2fc)))
genes = genes.sort_values(['q_value_exploratory','p_value_exploratory']).reset_index(drop=True)
hits = genes.loc[genes.candidate_exploratory].copy()
display(genes.head(20)); print('Exploratory candidates:',len(hits))
guide.to_csv(OUT/'guide_effects.csv',index_label='sgRNA')
genes.to_csv(OUT/'gene_results_exploratory.csv',index=False)
hits.to_csv(OUT/'candidate_genes_exploratory.csv',index=False)

## 7. Visualize candidates and inspect individual guides
Positive effects indicate enrichment in treatment; negative effects indicate depletion. Check that several guides support a candidate, rather than relying on one extreme guide.

In [ ]:
plotdata=genes.copy(); plotdata['minus_log10_q']=-np.log10(plotdata.q_value_exploratory.clip(lower=1e-300))
plt.figure(figsize=(8,5))
sns.scatterplot(data=plotdata,x='log2fc',y='minus_log10_q',hue='candidate_exploratory',alpha=.7)
plt.axhline(-np.log10(FDR_THRESHOLD),ls='--',color='grey')
for x in [-MIN_ABS_LOG2FC,MIN_ABS_LOG2FC]: plt.axvline(x,ls=':',color='grey')
for _,r in plotdata.head(6).iterrows(): plt.text(r.log2fc,r.minus_log10_q,r.Gene,fontsize=8)
plt.title('Exploratory gene effects — SIMULATED' if USE_DEMO else 'Exploratory gene effects')
plt.ylabel('-log10(exploratory BH q-value)'); plt.tight_layout()
plt.savefig(OUT/'gene_volcano_exploratory.png',dpi=160); plt.show()
selected=genes.head(12).Gene.tolist()
plt.figure(figsize=(8,6))
sns.stripplot(data=guide.reset_index().query('Gene in @selected'),x='log2fc',y='Gene',color='steelblue')
plt.axvline(0,color='grey'); plt.title('Individual guide effects for top-ranked genes')
plt.tight_layout(); plt.savefig(OUT/'guide_concordance.png',dpi=160); plt.show()
if USE_DEMO:
    truth_hits=set(truth.loc[truth.true_log2fc!=0,'Gene'])
    predicted=set(hits.Gene)
    print('Simulation-only recovery:',len(predicted&truth_hits),'/',len(truth_hits))
    print('Simulation-only false candidates:',len(predicted-truth_hits))
    print('These numbers do not measure performance on real data.')

## 8. MAGeCK: count-aware confirmation
MAGeCK tests guide counts and aggregates guide ranks to genes. Use **raw counts**, not the exploratory normalized table. The command below uses sample names and exports the analysis inputs. Apply consistent control definitions and investigate differences between methods.

Run the generated command in an environment where MAGeCK is installed, or run **MAGeCK test** in Galaxy and upload its `gene_summary.txt` below. Installation is intentionally not automated because MAGeCK has platform-dependent dependencies. Follow the official installation guide: https://sourceforge.net/p/mageck/wiki/installation/ . This notebook can execute MAGeCK if it already exists in the runtime.

For multiple conditions or covariates, use MAGeCK MLE with a design matrix instead of the simple command here. If comparing both enrichment and depletion, the two separate MAGeCK FDR columns are direction-specific; do not claim their union has a single 5% FDR.

In [ ]:
import shlex
mageck_counts=counts[['sgRNA','Gene']+samples]
mageck_path=OUT/'mageck_raw_counts.tsv'
mageck_counts.to_csv(mageck_path,sep='\t',index=False)
metadata.to_csv(OUT/'sample_metadata.csv',index=False)
command=['mageck','test','-k',str(mageck_path),'-t',','.join(trt),'-c',','.join(ctrl),
         '-n',str(OUT/'mageck'),'--norm-method','control' if control_ids else 'median']
if control_ids:
    (OUT/'control_guides.txt').write_text('\n'.join(control_ids)+'\n')
    command += ['--control-sgrna',str(OUT/'control_guides.txt')]
if PAIRED:
    command[command.index('-t')+1]=','.join(tcols)
    command[command.index('-c')+1]=','.join(ccols)
    command += ['--paired']
command_text=shlex.join(command)
(OUT/'run_mageck.sh').write_text('#!/usr/bin/env bash\nset -euo pipefail\n'+command_text+'\n')
print(command_text)
RUN_MAGECK = False  # Set True only after installing MAGeCK in this runtime.
if RUN_MAGECK:
    if shutil.which('mageck') is None: raise RuntimeError('Install MAGeCK or use Galaxy, then import results.')
    completed=subprocess.run(command,check=True,capture_output=True,text=True)
    (OUT/'mageck_execution.log').write_text(completed.stdout+'\n'+completed.stderr)
    version=subprocess.run(['mageck','--version'],capture_output=True,text=True)
    (OUT/'mageck_version.txt').write_text(version.stdout+version.stderr)
else:
    print('MAGeCK has NOT run. Exploratory statistics above are not MAGeCK results.')

In [ ]:
IMPORT_MAGECK = False  # True opens a Colab upload dialog for gene_summary.txt.
summary_path=OUT/'mageck.gene_summary.txt'
if IMPORT_MAGECK:
    from google.colab import files
    upload=files.upload()
    summaries=[p for p in upload if p.endswith('gene_summary.txt')]
    if len(summaries)!=1: raise ValueError('Upload exactly one MAGeCK gene_summary.txt.')
    shutil.copyfile(summaries[0],summary_path)
mageck_results=None
if summary_path.exists():
    mageck_results=pd.read_csv(summary_path,sep='\t')
    needed={'id','neg|fdr','pos|fdr','neg|lfc','pos|lfc'}
    if not needed.issubset(mageck_results): raise ValueError('Expected MAGeCK test/RRA gene_summary format.')
    if mageck_results.id.duplicated().any(): raise ValueError('Duplicate genes in imported summary.')
    if not set(mageck_results.id).issubset(set(counts.Gene)):
        raise ValueError('Imported results contain genes absent from the input library.')
    for col in ['neg|fdr','pos|fdr']:
        values=pd.to_numeric(mageck_results[col],errors='raise')
        if not np.isfinite(values).all() or not values.between(0,1).all(): raise ValueError('Invalid FDR values.')
    for col in ['neg|lfc','pos|lfc']:
        mageck_results[col]=pd.to_numeric(mageck_results[col],errors='raise')
        if not np.isfinite(mageck_results[col]).all(): raise ValueError('Nonfinite MAGeCK effect sizes.')
    neutral_genes=set(annotation.loc[annotation.index.intersection(control_ids)])
    m= mageck_results.loc[~mageck_results.id.isin(neutral_genes)].copy()
    negative=m.loc[(m['neg|fdr']<=FDR_THRESHOLD)&(m['neg|lfc']<=-MIN_ABS_LOG2FC)]
    positive=m.loc[(m['pos|fdr']<=FDR_THRESHOLD)&(m['pos|lfc']>=MIN_ABS_LOG2FC)]
    negative.to_csv(OUT/'mageck_depleted_candidates.csv',index=False)
    positive.to_csv(OUT/'mageck_enriched_candidates.csv',index=False)
    display(negative.head(15),positive.head(15))
    print('Verify imported summary corresponds to the exact samples, contrast, normalization and library.')
else:
    print('No MAGeCK summary imported: count-aware gene inference remains to be completed.')

## 9. Optional pathway enrichment from a local GMT file
Upload curated gene sets matching your species and identifier type, such as symbols or Ensembl IDs. Use genes eligible for the screen analysis as the background, not all genes in the genome. This avoids a major source of enrichment bias. Demo gene IDs are artificial and do not support real pathway interpretation.

This section uses a hypergeometric over-representation test with BH correction. It is exploratory and sensitive to shortlist thresholds. Review correlated pathways and compare with a ranked-list method where suitable.

In [ ]:
RUN_PATHWAYS = False
PATHWAY_DIRECTION = 'depleted'  # or enriched
PATHWAY_SOURCE = 'mageck'  # or exploratory, explicitly labelled
if RUN_PATHWAYS:
    if USE_DEMO: raise ValueError('Real pathway analysis requires real gene identifiers.')
    if PATHWAY_DIRECTION not in {'depleted','enriched'}: raise ValueError('Choose depleted/enriched.')
    if PATHWAY_SOURCE=='mageck':
        if mageck_results is None: raise ValueError('Import MAGeCK first.')
        background=set(m.id)
        chosen=set(negative.id if PATHWAY_DIRECTION=='depleted' else positive.id)
    elif PATHWAY_SOURCE=='exploratory':
        background=set(genes.Gene)
        chosen=set(hits.loc[hits.direction==PATHWAY_DIRECTION,'Gene'])
    else: raise ValueError('Choose mageck/exploratory.')
    if not chosen: raise ValueError('No candidates for pathway testing.')
    from google.colab import files
    uploaded=files.upload()
    gmt=[p for p in uploaded if p.endswith('.gmt')]
    if len(gmt)!=1: raise ValueError('Upload one GMT file.')
    rows=[]; seen=set()
    for line in Path(gmt[0]).read_text().splitlines():
        parts=line.split('\t')
        if len(parts)<3: continue
        if parts[0] in seen: raise ValueError('Duplicate pathway name.')
        seen.add(parts[0]); members=set(parts[2:])&background
        if not 10<=len(members)<=500: continue
        overlap=chosen&members
        p=stats.hypergeom.sf(len(overlap)-1,len(background),len(members),len(chosen))
        rows.append({'pathway':parts[0], 'tested_members':len(members), 'overlap':len(overlap),
                     'p_value':p,'genes':';'.join(sorted(overlap))})
    enrichment=pd.DataFrame(rows)
    if enrichment.empty: raise ValueError('No matching gene sets; check identifiers/species.')
    enrichment['q_value']=bh_adjust(enrichment.p_value)
    enrichment=enrichment.sort_values('q_value')
    enrichment.to_csv(OUT/f'pathways_{PATHWAY_SOURCE}_{PATHWAY_DIRECTION}.csv',index=False)
    shutil.copyfile(gmt[0],OUT/'pathway_gene_sets.gmt')
    display(enrichment.head(20))
else:
    print('Pathway analysis skipped; enable after obtaining real gene sets and candidates.')

## 10. Candidate review and validation proposal
For each candidate, document independent guide support, biological plausibility, known essentiality, copy-number effects, off-target concerns, and any evidence from an untreated/time-zero contrast. Separate general growth genes from trait-specific effects.

The editable evidence table contains blank fields for your literature review. Independent perturbations, rescue experiments and an orthogonal trait readout can strengthen follow-up evidence. Do not label a candidate as a confirmed causal gene from this screen alone.

**Final report outline:** research question → data provenance and design → QC → prespecified analysis → gene results → biological interpretation → limitations → proposed validation. Include all tested genes, not only hits.

In [ ]:
review=hits[['Gene','direction','log2fc','q_value_exploratory','n_guides','guide_concordance']].copy()
for field in ['literature_reference','known_essentiality','copy_number_concern','trait_specific_evidence',
              'independent_validation_proposal','review_notes']:
    review[field]=''
review.to_csv(OUT/'candidate_evidence_template.csv',index=False)
# Save user-supplied count inputs as well as the design and environment.
counts.to_csv(OUT/'input_counts.csv',index=False)
config={'use_demo':USE_DEMO,'trait':TRAIT,'species':SPECIES,'paired':PAIRED,'seed':SEED,
        'min_control_count':MIN_CONTROL_COUNT,'min_control_fraction':MIN_CONTROL_FRACTION,
        'min_guides_per_gene':MIN_GUIDES_PER_GENE,'fdr_threshold':FDR_THRESHOLD,
        'min_abs_log2fc':MIN_ABS_LOG2FC,'min_guide_concordance':MIN_GUIDE_CONCORDANCE,
        'normalization':'control_median_ratio' if control_ids else 'median_ratio',
        'exploratory_method':'paired_t_on_gene_median_logcounts' if PAIRED else 'welch_t_on_gene_median_logcounts'}
(OUT/'analysis_config.json').write_text(json.dumps(config,indent=2))
import importlib.metadata
versions={p:importlib.metadata.version(p) for p in requirements.values()}
versions['python']=platform.python_version()
(OUT/'environment_versions.json').write_text(json.dumps(versions,indent=2))
hash_files=['input_counts.csv','sample_metadata.csv']
(OUT/'input_sha256.json').write_text(json.dumps({f:hashlib.sha256((OUT/f).read_bytes()).hexdigest() for f in hash_files},indent=2))
report=f"""# Screen analysis summary
Data mode: {'SIMULATED — no biological discoveries' if USE_DEMO else 'REAL DATA'}
Trait: {TRAIT}
Guides: {len(raw)} input; {len(filtered)} retained.
Eligible tested genes: {len(genes)}.
Exploratory candidates: {len(hits)}.
MAGeCK summary available: {mageck_results is not None}.
Exploratory tests use biological samples, not guides as independent replicates.
Interpretation: positive = enriched in treatment; negative = depleted.
Complete literature review, compare fitness effects and validate before causal claims.
"""
(OUT/'analysis_summary.md').write_text(report)
archive=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print(report); print('Results ZIP:',archive)
DOWNLOAD_RESULTS = True
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print('Jupyter users: download the ZIP from the file browser.')

## References and reporting notes
- MAGeCK original method: Li et al. (2014), *Genome Biology* 15:554. https://doi.org/10.1186/s13059-014-0554-4
- Official usage (sample selection, normalization and paired analysis): https://sourceforge.net/p/mageck/wiki/usage/
- Official output-column definitions: https://sourceforge.net/p/mageck/wiki/output/
- Official installation instructions: https://sourceforge.net/p/mageck/wiki/installation/

Record dataset accession, library version, organism, cell model, selection endpoint, biological replicate definition, and all upstream count-generation settings in your report. Keep donor/batch identities and potential confounders in the study design. Statistical significance indicates evidence under a model; it does not establish causation or clinical relevance.

For a GitHub repository, include this notebook, a README describing the design, environment versions, non-sensitive example inputs and a data-access statement. Do not upload restricted individual-level data.